# rl-perf Quick Start

This notebook demonstrates how to use rl-perf to analyze RL training performance.

We'll cover:
1. Loading model and hardware configs
2. Running a basic targets analysis
3. Interpreting the results
4. Configuring a custom model

In [ ]:
from rl_perf.config import load_model_config, load_hardware_config, RLConfig, ParallelismConfig
from rl_perf.model import RLPerformanceModel
from rl_perf.report import format_table

## 1. Load Configs
rl-perf ships with 5 demo model configs and 2 hardware configs.

In [ ]:
# Load a model and hardware config
model = load_model_config("../configs/models/llama3_1_8b.yaml")
hw = load_hardware_config("../configs/hardware/ascend_910c.yaml")

print(f"Model: {model.name}")
print(f"  Hidden size: {model.hidden_size}")
print(f"  Layers: {model.num_layers}")
print(f"  Attention: {model.get_layers()[0].attention}")
print(f"  FFN: {model.get_layers()[0].ffn}")
print(f"\nHardware: {hw.name}")
print(f"  Peak TFLOPS: {hw.peak_tflops_bf16}")
print(f"  HBM: {hw.hbm_capacity_gb} GB (usable: {hw.usable_hbm_gb:.1f} GB)")

## 2. Define RL Training Parameters

In [ ]:
# RL training configuration
rl_cfg = RLConfig(
    total_prompts=10000,          # Total prompts in one epoch
    group_size=8,                 # GRPO: responses per prompt
    avg_prompt_len=512,           # Average input length (tokens)
    avg_response_len=2048,        # Average output length (tokens)
    max_response_len=4096,        # Max output (for long-tail modeling)
    train_micro_batch_size=4,
    gradient_accumulation_steps=4,
    gen_batch_size=64,
)

print(f"Total responses per epoch: {rl_cfg.total_responses:,}")
print(f"Total tokens to generate: {rl_cfg.total_responses * rl_cfg.avg_response_len:,}")

## 3. Run Targets Analysis

Set up parallelism configs for generation and training phases, then derive TPS targets.

In [ ]:
# Parallelism: 64 devices, TP=8
gen_parallel = ParallelismConfig(tp=8, dp=8)        # Generation: TP=8, DP=8
train_parallel = ParallelismConfig(tp=8, dp=8)      # Training: TP=8, DP=8

perf = RLPerformanceModel(model, hw)
report = perf.derive_targets(
    total_devices=64,
    rl_cfg=rl_cfg,
    gen_parallel=gen_parallel,
    train_parallel=train_parallel,
    time_budget_hours=24,
)

print(format_table(report))

## 4. Compare Parallelism Strategies

Let's see how different TP/PP/DP combinations affect performance for a larger model.

In [ ]:
# Load Qwen2.5 72B — needs PP for memory
model_72b = load_model_config("../configs/models/qwen2_5_72b.yaml")
perf_72b = RLPerformanceModel(model_72b, hw)

configs = [
    ("TP=8 PP=1 DP=16", ParallelismConfig(tp=8, pp=1, dp=16)),
    ("TP=8 PP=2 DP=8",  ParallelismConfig(tp=8, pp=2, dp=8)),
    ("TP=8 PP=4 DP=4",  ParallelismConfig(tp=8, pp=4, dp=4)),
]

print(f"{'Config':<20} {'Epoch(h)':>8} {'Bottleneck':<12} {'Train Mem':>10} {'Feasible':>8}")
print("-" * 65)
for name, p in configs:
    gen_p = ParallelismConfig(tp=8, dp=128 // 8)
    r = perf_72b.derive_targets(128, rl_cfg, gen_p, p, 24)
    print(f"{name:<20} {r.epoch_time_hours:>8.2f} {r.bottleneck:<12} "
          f"{r.memory.total_train_gb:>8.1f}GB {'OK' if r.memory.train_feasible else 'OOM':>8}")

## 5. Sensitivity Analysis: Group Size

How does GRPO group_size affect epoch time?

In [ ]:
print(f"{'Group Size':>10} {'Epoch(h)':>10} {'Gen TPS':>12} {'Train TPS':>12} {'Bottleneck':<12}")
print("-" * 60)
for gs in [4, 8, 16, 32]:
    rl_gs = rl_cfg.model_copy(update={"group_size": gs})
    r = perf.derive_targets(64, rl_gs, gen_parallel, train_parallel, 24)
    print(f"{gs:>10} {r.epoch_time_hours:>10.2f} {r.gen_tps_target:>12,.0f} "
          f"{r.train_tps_target:>12,.0f} {r.bottleneck:<12}")

## 6. Configure a Custom Model

Copy `configs/models/_template.yaml`, modify parameters, and analyze.

In [ ]:
from rl_perf.config import ModelConfig, LayerConfig

# Define inline (or load from YAML)
custom_model = ModelConfig(
    name="Custom-20B-MoE",
    hidden_size=4096,
    vocab_size=128000,
    num_layers=48,
    dtype="bf16",
    default_layer=LayerConfig(
        attention="GQA",
        num_heads=32,
        num_kv_heads=8,
        head_dim=128,
        ffn="MoE",
        num_experts=64,
        num_shared_experts=2,
        top_k=6,
        expert_intermediate_size=2048,
        shared_intermediate_size=11008,
        intermediate_size=11008,
    ),
)

perf_custom = RLPerformanceModel(custom_model, hw)
gen_p = ParallelismConfig(tp=8, dp=16)
train_p = ParallelismConfig(tp=8, pp=2, dp=4, ep=8)

r = perf_custom.derive_targets(128, rl_cfg, gen_p, train_p, 24)
print(format_table(r))